# DIAGNOSTIC TOOL

In [1]:
import numpy as np
import pandas as pd
import skimage as ski
import matplotlib.pyplot as plt
import tifffile
import zarr
import os
import xml.etree.ElementTree as ET
import napari
from importlib import reload

In [2]:
import load_assess_image as load_assess
reload(load_assess)

<module 'load_assess_image' from 'r:\\Thu\\From_2026\\6. Other\\DL_Summer_2026\\notebook\\load_assess_image.py'>

In [3]:
import qc_widget
reload(qc_widget)

<module 'qc_widget' from 'r:\\Thu\\From_2026\\6. Other\\DL_Summer_2026\\notebook\\qc_widget.py'>

In [5]:
main_dir=rf'D:\thu\HTAN\images\ILC\level_2'
file_name='OHSU_TMA1_005-A7.ome.tif'
# image_test=load_assess.AssessImage(main_dir=main_dir,
                    # filename=file_name)

# QC Stats

In [54]:

from concurrent.futures import ThreadPoolExecutor   
def process_one(args):
    main_dir, filename = args
    image = load_assess.AssessImage(main_dir, filename)
    df = image.get_stats(resolution_level=2)
    image.loader.tif.close()
    image.loader.image_store.close()
    return filename, df




In [9]:
file_name, df = process_one([main_dir, list_file_with_pRB[1]])

NameError: name 'process_one' is not defined

In [20]:
image = load_assess.AssessImage(main_dir, list_file_with_pRB[2])
image.view_images(biomarkers_of_interest=['dna(5)','dna(4)','dna'],resolution_level=0)


Finding biomarkers: 100%|██████████| 39/39 [00:00<00:00, 60.58it/s]


Viewer(mouse_move_callbacks=[], mouse_wheel_callbacks=[<function dims_scroll at 0x000001D231B75440>], mouse_drag_callbacks=[<function drag_to_zoom at 0x000001D231B75580>], mouse_double_click_callbacks=[<function double_click_to_zoom at 0x000001D231B754E0>], camera=Camera(center=(0.0, 1362.5, 1419.5), zoom=0.33107116654438734, angles=(0.0, 0.0, 0.0), perspective=0.0, mouse_pan=True, mouse_zoom=True, orientation=(<DepthAxisOrientation.TOWARDS: 'towards'>, <VerticalAxisOrientation.DOWN: 'down'>, <HorizontalAxisOrientation.RIGHT: 'right'>)), cursor=Cursor(position=(1.0, 1.0), viewbox=None, scaled=True, size=1.0, style=<CursorStyle.STANDARD: 'standard'>), dims=Dims(ndim=2, ndisplay=2, order=(0, 1), axis_labels=('-2', '-1'), rollable=(True, True), range=(RangeTuple(start=0.0, stop=2725.0, step=1.0), RangeTuple(start=0.0, stop=2839.0, step=1.0)), margin_left=(0.0, 0.0), margin_right=(0.0, 0.0), point=(1362.0, 1419.0), units=(<Unit('pixel')>, <Unit('pixel')>), last_used=0), grid=GridCanvas(str

In [19]:
 list_file_with_pRB[2]

'LSP12022-B5.ome.tif'

# Multiple Images

In [15]:
list_file_with_pRB=[]
for file in os.listdir(main_dir):
    if "QC" not in file and "MIP_cyto" not in file and "mask" not in file and "test_QUALIFAI" not in file:
        image_object= load_assess.AssessImage(main_dir,file)
        if 'pRB' in list(image_object.biomarker_channel):
            list_file_with_pRB.append(file)
            # print(image_object.biomarker_channel)


In [16]:
list_file_with_pRB

['LSP12022-A7.ome.tif',
 'LSP12022-B1.ome.tif',
 'LSP12022-B5.ome.tif',
 'LSP12022-B9.ome.tif',
 'LSP12022-C3.ome.tif',
 'LSP12022-C8.ome.tif',
 'LSP12022-D2.ome.tif',
 'LSP12022-F3.ome.tif',
 'LSP12022-F7.ome.tif',
 'LSP12022-H2.ome.tif',
 'LSP12022-H6.ome.tif',
 'LSP12022-H8.ome.tif',
 'OHSU_TMA1_004-A7.ome.tif',
 'OHSU_TMA1_004-B1.ome.tif',
 'OHSU_TMA1_004-B5.ome.tif',
 'OHSU_TMA1_004-B9.ome.tif',
 'OHSU_TMA1_004-C3.ome.tif',
 'OHSU_TMA1_004-C8.ome.tif',
 'OHSU_TMA1_004-D2.ome.tif',
 'OHSU_TMA1_004-F3.ome.tif',
 'OHSU_TMA1_004-F7.ome.tif',
 'OHSU_TMA1_004-H2.ome.tif',
 'OHSU_TMA1_004-H6.ome.tif',
 'OHSU_TMA1_004-H8.ome.tif',
 'OHSU_TMA1_005-A7.ome.tif',
 'OHSU_TMA1_005-B1.ome.tif',
 'OHSU_TMA1_005-B5.ome.tif',
 'OHSU_TMA1_005-B9.ome.tif',
 'OHSU_TMA1_005-C3.ome.tif',
 'OHSU_TMA1_005-C8.ome.tif',
 'OHSU_TMA1_005-D2.ome.tif',
 'OHSU_TMA1_005-F3.ome.tif',
 'OHSU_TMA1_005-F7.ome.tif',
 'OHSU_TMA1_005-H2.ome.tif',
 'OHSU_TMA1_005-H6.ome.tif',
 'OHSU_TMA1_005-H8.ome.tif',
 'OHSU_TMA1_010-

In [64]:
# CHECK CD163 for segmentation

            
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = pool.map(process_one, [(main_dir, i) for i in list_file_with_pRB[:26]])
    results = dict(futures)

In [16]:
combined_qc = pd.concat(results, names=['filename']).reset_index(level=0)
combined_qc.to_csv('qc_pRB_ki67_files_test.csv')

In [17]:
biomarker=['aSMA','CD45','CD163','pRb','Ki67']
combined_qc[combined_qc['biomarker'].isin(biomarker)]['filename'].unique().tolist()

['LSP12022-A7.ome.tif',
 'LSP12022-B1.ome.tif',
 'LSP12022-B5.ome.tif',
 'LSP12022-B9.ome.tif',
 'LSP12022-C3.ome.tif',
 'LSP12022-C8.ome.tif',
 'LSP12022-D2.ome.tif',
 'LSP12022-F3.ome.tif',
 'LSP12022-F7.ome.tif',
 'LSP12022-H2.ome.tif',
 'LSP12022-H6.ome.tif',
 'LSP12022-H8.ome.tif',
 'OHSU_TMA1_004-A7.ome.tif',
 'OHSU_TMA1_004-B1.ome.tif',
 'OHSU_TMA1_004-B5.ome.tif',
 'OHSU_TMA1_004-B9.ome.tif',
 'OHSU_TMA1_004-C3.ome.tif',
 'OHSU_TMA1_004-C8.ome.tif',
 'OHSU_TMA1_004-D2.ome.tif',
 'OHSU_TMA1_004-F3.ome.tif',
 'OHSU_TMA1_004-F7.ome.tif',
 'OHSU_TMA1_004-H2.ome.tif',
 'OHSU_TMA1_004-H6.ome.tif',
 'OHSU_TMA1_004-H8.ome.tif',
 'OHSU_TMA1_005-A7.ome.tif',
 'OHSU_TMA1_005-B1.ome.tif']

In [11]:
channel=['dna','aSMA','CD45','CD163','pRb','Ki67','CD163']
image_test=load_assess.AssessImage(main_dir,filename= 'LSP12022-B9.ome.tif')
viewer = image_test.view_images(biomarkers_of_interest=channel, resolution_level=0)


Finding biomarkers: 100%|██████████| 39/39 [00:01<00:00, 30.20it/s]


# QC Manually

In [12]:
qc_dir = os.path.join(main_dir, 'QC')
os.makedirs(qc_dir, exist_ok=True)
print(qc_dir) 

D:\thu\HTAN\images\ILC\level_2\QC


In [20]:
%gui qt

channel=['dna','aSMA','CD45','CD163','pRb','Ki67','CD163']
test=combined_qc[combined_qc['filename']=='LSP12022-B9.ome.tif']
flagged = test[test['flag'] != 'OK']



In [40]:
import napari

filename = 'LSP12022-B9.ome.tif'
assessor = load_assess.AssessImage(main_dir, filename)
flagged  = assessor.get_stats(resolution_level=2).query("flag != 'OK'")

viewer = napari.Viewer()  # create a fresh viewer here
widget = qc_widget.QCWidget(viewer, assessor, flagged,
                            save_path=f'{filename}_qc_decisions.csv',
                            resolution_level=0)
viewer.window.add_dock_widget(widget, name='QC Review')


In [36]:
assessor.biomarker_channel.keys()

dict_keys(['dna', 'control488nm', 'control555nm', 'control647nm', 'dna(2)', 'RNApol2-pS2', 'pERK', 'p53', 'dna(3)', 'Ki67', 'panCK', 'CD45', 'dna(4)', 'Ecad', 'aSMA', 'Vim', 'dna(5)', 'EGFR', 'pRb', 'p21', 'dna(6)', 'CD20', 'PD1', 'dna(7)', 'PCNA', 'ER', 'HER2', 'dna(8)', 'CK1-4', 'CK19', 'CK17', 'dna(9)', 'LamAC', 'AR', 'gH2AX', 'dna(10)', 'PR', 'HLA-A', 'CK5'])